# [SK 04.5 - RESPONSES Agent using `Invoke` (with plugins)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.open_ai.azure_responses_agent.azureresponsesagent?view=semantic-kernel-python)
Documentation [here](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/responses-agent?pivots=programming-language-python).

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")

agent_name                  = "my_response_agent"
instructions                = "you are a clever agent"

plugin_name                 = "Lights"

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [2]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler())
logging.getLogger().setLevel(logging.ERROR) # or logging.DEBUG to see more details

# Native Plugin

In [3]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Creating an [AzureResponsesAgent](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.open_ai.azure_responses_agent.azureresponsesagent?view=semantic-kernel-python) with plugins

## Creating an AzureResponsesAgent requires first creating a client to be able to talk a remote Azure OpenAI service

In [4]:
from semantic_kernel.agents import AzureResponsesAgent
from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings

# deployment name can be implicit if the following environment variables are set
os.environ['AZURE_OPENAI_ENDPOINT'] = os.environ['AZURE_OPENAI_ENDPOINT'] # variable already exists, same name
os.environ['AZURE_OPENAI_API_KEY']  = os.environ['AZURE_OPENAI_API_KEY'] # variable already exists, same name
os.environ['AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME'] = os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'] # variable already exists, different name

# Set up the client and model using Azure OpenAI Resources
client = AzureResponsesAgent.create_client()

AzureOpenAISettings()

AzureOpenAISettings(env_file_path=None, env_file_encoding='utf-8', chat_deployment_name='gpt-4o', responses_deployment_name='gpt-4o', text_deployment_name='gpt-35-turbo-instruct', embedding_deployment_name='text-embedding-ada-002', text_to_image_deployment_name=None, audio_to_text_deployment_name=None, text_to_audio_deployment_name=None, realtime_deployment_name=None, endpoint=AnyUrl('https://mmoaiswc-01.openai.azure.com/'), base_url=None, api_key=SecretStr('**********'), api_version='2025-04-01-preview', token_endpoint='https://cognitiveservices.azure.com/.default')

## Create the AzureResponsesAgent instance using the client and the model

In [5]:
agent = AzureResponsesAgent(
    ai_model_id=AzureOpenAISettings().responses_deployment_name,
    client=client,
    instructions=instructions,
    name=agent_name,
    plugins=[LightsPlugin()]
)

agent

AzureResponsesAgent(arguments=None, description=None, id='f29c1565-378c-4a3f-9f35-7278b6d8fe5f', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x7e061852dd30>, plugins={'LightsPlugin': KernelPlugin(name='LightsPlugin', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='LightsPlugin', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=F

# Using an OpenAIResponsesAgent
The OpenAI Responses API supports optional remote storage of conversations. By default, when using a ResponsesAgentThread, responses are stored remotely. This enables the use of the Responses API's previous_response_id for maintaining context across invocations.<br/>

Each conversation is treated as a thread, identified by a unique string ID. All interactions with your OpenAIResponsesAgent are scoped to this thread identifier.<br/>

The underlying mechanics of the Responses API thread are abstracted by the ResponsesAgentThread class, which implements the AgentThread interface.<br/>

The OpenAIResponsesAgent currently only supports threads of type ResponsesAgentThread.<br/>

You can invoke the OpenAIResponsesAgent without specifying an AgentThread, to start a new thread and a new AgentThread will be returned as part of the response.

## Use and intermediate_steps callback function
This won't be used, I report here just as a reminder

In [6]:
# This callback function will be called for each intermediate message,
# which will allow one to handle FunctionCallContent and FunctionResultContent.
# If the callback is not provided, the agent will return the final response
# with no intermediate tool call steps.

from semantic_kernel.contents.chat_message_content import ChatMessageContent

async def handle_intermediate_steps(message: ChatMessageContent) -> None:
    for item in message.items or []:
        if isinstance(item, FunctionResultContent):
            print(f"Function Result:> {item.result} for function: {item.name}")
        elif isinstance(item, FunctionCallContent):
            print(f"Function Call:> {item.name} with arguments: {item.arguments}")
        else:
            print(f"{item}")

## Call the agent with `invoke()`

In [7]:
# Clean the history - Just for testing purposes, set clear_history as True
# When we clean the history, we must make sure to reset also the plugin status, which otherwise stores its status
from semantic_kernel.agents import ChatHistoryAgentThread

thread:ChatHistoryAgentThread = None

clear_history:bool = False # set as True for testing purposes

if clear_history:
    thread: ChatHistoryAgentThread = None
    
    agent = AzureResponsesAgent(
        ai_model_id=AzureOpenAISettings().responses_deployment_name,
        client=client,
        instructions=instructions,
        name=agent_name,
        plugins=[LightsPlugin()]
    )

In [8]:
# Create a thread for the agent
# If no thread is provided, a new thread will be
# created and returned with the initial response

from semantic_kernel.contents import AuthorRole, FunctionResultContent

user_inputs = ["Hello", "Please toggle the porch light", "What's the status of all lights?", "Thank you"]

try:
    for user_input in user_inputs:
        print(f"# {AuthorRole.USER}: '{user_input}'")
        async for response in agent.invoke(
            messages=user_input,
            thread=thread,
            # on_intermediate_message=handle_intermediate_steps,
        ):
            thread = response.thread
            print(f"# {response.name}: {response.content}")
finally:
    if thread:
        await thread.delete()
        print(f"\nthread <{thread.id}> has been deleted.")

# AuthorRole.USER: 'Hello'
# my_response_agent: Hi there! How can I assist you today?
# AuthorRole.USER: 'Please toggle the porch light'
# my_response_agent: The porch light has been toggled on. If you need anything else, let me know!
# AuthorRole.USER: 'What's the status of all lights?'
# my_response_agent: Here's the current status of the lights:

- **Table Lamp**: Off
- **Porch Light**: On
- **Chandelier**: Off

Let me know if you need further assistance!
# AuthorRole.USER: 'Thank you'
# my_response_agent: You're welcome! If you have any more questions or need help, just let me know. Have a great day!

thread <resp_064da6330de12f4e00695fbcf134648197a0ea64f01cd1b29b> has been deleted.
